# Attention 메커니즘 직접 구현 - 실습 코드 1: 완전한 Self-Attention 구현

- Tutorial ID: `expand-attention-from-scratch`
- Tutorial: Attention 메커니즘 직접 구현
- Section ID: `expand-attention-from-scratch-code-1`
- Section: 실습 코드 1: 완전한 Self-Attention 구현

이 노트북은 Transformer의 핵심인 **Self-Attention**을 별도 딥러닝 프레임워크 없이 numpy만으로 직접 구현하면서, "왜 이렇게 계산하는가"를 하나씩 짚어보는 실습 자료입니다. 수식을 외우기보다는, 아주 작은 예제를 손으로 따라가며 **벡터가 어떤 shape으로 변환되고 숫자가 어떤 의미를 갖는지** 확인하는 데 집중해주세요.

> 💡 이 버전은 원본 실습 코드에 있던 들여쓰기 오류(그대로는 실행되지 않는 상태였습니다)를 수정하고, 개념을 단계별로 풀어서 설명하는 마크다운 셀과 예제를 추가했습니다.

In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: 완전한 Self-Attention 구현
#
# 이 코드는 "정답을 한 번 실행하고 끝내는" 용도가 아니라, Self-Attention의 수식이
# 실제 numpy 배열 연산으로 바뀌는 과정을 한 줄씩 따라가며 확인하기 위한 실습
# 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고, 그로부터 attention score가 어떻게
#      계산되는지 shape 변화를 눈으로 추적한다.
#   2) 내적(dot product) 점수가 softmax를 거쳐 "합이 1인 확률분포"로 바뀌는
#      과정을 직접 계산해본다.
#   3) 미래 토큰을 -inf(사실상 매우 작은 수)로 막았을 때, softmax를 통과한
#      확률이 실제로 0이 되는지 확인한다.
#   4) 여러 개의 attention head를 합치는 Multi-Head Attention의 구조와 shape
#      변화를 추적한다.
#
# 읽는 순서:
#   1) 먼저 이 노트북에서 쓰는 용어(Query/Key/Value, d_model, d_k 등)를 설명하는
#      마크다운 셀을 읽습니다.
#   2) 아주 작은 예제(단어 3개 x 4차원)로 attention 계산을 손으로 따라가듯
#      한 단계씩(점수 계산 → 스케일링 → softmax → 가중합) 실행해봅니다.
#   3) 위 단계들을 하나의 함수 scaled_dot_product_attention으로 정리한 뒤,
#      같은 결과가 나오는지 직접 비교해봅니다.
#   4) mask를 추가해 "미래를 보지 못하게 막는다"는 것이 실제로 숫자에 어떤
#      영향을 주는지 확인합니다.
#   5) 마지막으로 여러 head를 사용하는 MultiHeadAttention 클래스를 만들고,
#      입력 X가 각 단계를 거치며 어떤 shape으로 변하는지 출력해봅니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 "shape이 어떻게 바뀌는지"와 "정보(값)가 어느
#     방향으로 흘러가는지"를 눈으로 따라가 보세요.
#   - 이 노트북은 딥러닝 프레임워크 없이 numpy와 matplotlib만 사용합니다.
#     실제 프레임워크(PyTorch 등)에서는 배치(batch) 차원이 추가되고 내부적으로
#     더 최적화된 연산을 사용하지만, 핵심 원리는 동일합니다.
# ============================================================

## 1. Self-Attention이 왜 필요할까요?

다음 문장을 예로 들어보겠습니다.

> "그 동물은 길을 건너지 않았다. 왜냐하면 **그것**이 너무 피곤했기 때문이다."

여기서 "**그것**"이 가리키는 대상은 "동물"일까요, "길"일까요? 사람은 문맥을 보고 당연히 "동물"이라는 것을 알 수 있지만, 컴퓨터(모델) 입장에서는 "그것"이라는 단어 하나만 봐서는 정답을 알 수 없습니다. **문장 속 다른 단어들을 함께 참고해야만** "그것 = 동물"이라는 것을 알아낼 수 있습니다.

**Self-Attention**은 바로 이 역할을 합니다. 문장 속 각 단어가 "나를 잘 이해하려면 다른 어떤 단어들을 얼마나 참고해야 할까?"를 계산하는 방법입니다. 예를 들어

- "그것"이라는 단어는 → "동물"이라는 단어는 **많이** 참고하고, "길"이라는 단어는 **적게** 참고하도록 가중치(weight)를 계산합니다.

이렇게 계산한 가중치만큼 다른 단어들의 정보를 섞어 넣는 것이 Self-Attention의 핵심입니다. 이름에 "Self"가 붙은 이유는, 잠시 후에 볼 질의(Query)·키(Key)·값(Value)이 모두 **같은 문장(입력)** 에서 나오기 때문입니다. (참고로 번역 모델처럼 서로 다른 두 문장 사이에서 attention을 계산하면 이것은 Cross-Attention이라고 부릅니다.)

## 2. Query, Key, Value를 비유로 이해하기

Self-Attention은 "각 단어가 다른 단어를 얼마나 참고할지"를 세 가지 벡터로 계산합니다. 도서관에서 자료를 검색하는 상황에 비유해 보겠습니다.

| 개념 | 도서관 비유 | 의미 |
|---|---|---|
| **Query (질의)** | 내가 검색창에 입력하는 검색어 | "나는 지금 이런 정보가 필요해!" |
| **Key (키)** | 각 책의 표지에 붙어 있는 색인 태그 | "나는 이런 내용을 담고 있어!" |
| **Value (값)** | 책에 담긴 실제 내용 | 실제로 전달되는 정보 |

검색을 하면, 내 **Query**와 가장 잘 맞는 **Key**를 가진 책일수록 더 큰 비중으로 참고하게 되고, 그 책의 **Value**(실제 내용)를 더 많이 가져오게 됩니다. Self-Attention도 똑같습니다.

1. 문장 속 모든 단어가 각자 자신만의 Query, Key, Value 벡터를 만듭니다.
2. 한 단어의 Query와 다른 모든 단어의 Key를 비교(내적)해서 "얼마나 관련 있는지" 점수를 냅니다.
3. 이 점수를 softmax를 이용해 "비율(합이 1인 가중치)"로 바꿉니다.
4. 그 비율만큼 각 단어의 Value를 섞어서 최종 결과를 만듭니다.

정리하면 수식은 다음과 같습니다.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

지금은 이 수식이 낯설어도 괜찮습니다. 아래에서 이 수식을 한 조각씩 뜯어서 직접 계산해보면서 자연스럽게 익혀보겠습니다.

### 왜 내적(dot product)이 "유사도"가 될까요?

Query와 Key가 "얼마나 관련 있는지" 점수를 낼 때는 두 벡터의 **내적(dot product)** 을 사용합니다. 두 벡터가 비슷한 방향을 가리킬수록(=비슷한 의미/특성을 가질수록) 내적 값이 커진다는 성질을 이용하는 것입니다. 아래 코드로 간단히 확인해봅시다.

In [ ]:
import numpy as np

# 두 벡터가 "비슷한 방향"일 때와 "다른 방향"일 때 내적 값을 비교해봅니다.
a = np.array([1.0, 1.0, 0.0])             # 기준 벡터
b_similar = np.array([0.9, 1.0, 0.1])     # a와 방향이 비슷한 벡터
b_different = np.array([-1.0, 0.0, 1.0])  # a와 방향이 많이 다른 벡터

print("a · b_similar   =", a @ b_similar)    # 값이 큼   → "많이 관련 있다"
print("a · b_different =", a @ b_different)  # 값이 작음(또는 음수) → "관련이 적다"

# 즉 "내적이 크다" = "두 벡터가 비슷한 방향을 향한다" = "Query가 찾는 정보와 Key가 관련이 깊다"
# 라고 해석할 수 있습니다. Self-Attention은 이 성질을 그대로 이용해서 "관련도 점수"를 만듭니다.

## 3. softmax: 점수를 "확률처럼 보이는 가중치"로 바꾸기

내적으로 구한 점수(score)는 그냥 실수(−무한대 ~ +무한대)일 뿐, 아직 "비율"은 아닙니다. 이 점수를 아래 조건을 만족하는 **가중치**로 바꿔주는 함수가 바로 **softmax**입니다.

- 모든 값이 0 이상이다 (음수가 없다)
- 모든 값을 더하면 1이 된다 (비율로 해석할 수 있다)
- 원래 점수가 클수록 더 큰 비중을 차지한다 (순서는 그대로 유지된다)

수식으로는 다음과 같이 씁니다.

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

코드에서 `x - x_max`를 먼저 빼주는 이유는 **수치적으로 안정적으로 계산**하기 위해서입니다. $e^x$는 x가 조금만 커도 매우 큰 수가 되어 오버플로(overflow)가 날 수 있는데, 모든 값에서 최댓값을 빼주면 최종 결과(비율)는 수학적으로 완전히 동일하면서 지수 계산에 사용되는 값의 범위만 작아집니다. (가장 큰 값이 $e^0 = 1$이 되므로 안전합니다.)

In [ ]:
def softmax(x, axis=-1):
    """
    입력 x의 지정된 축(axis)을 기준으로 softmax를 계산합니다.
    (기본값 axis=-1은 "가장 마지막 차원"을 의미합니다. 2차원 배열이라면
     "각 행(row)마다 따로" softmax를 적용한다는 뜻입니다.)

    x: 임의의 실수 값들을 담은 배열 (attention에서는 QK^T로 구한 score)
    axis: softmax를 적용할 축
    """
    # 1) 수치 안정성을 위해 각 행에서 최댓값을 빼줍니다.
    #    (지수함수 exp()의 오버플로를 막기 위한 표준적인 트릭입니다.)
    x_max = np.max(x, axis=axis, keepdims=True)

    # 2) e^(x - x_max)를 계산합니다. 최댓값을 뺐기 때문에 결과는 항상 0~1 사이입니다.
    exp_x = np.exp(x - x_max)

    # 3) 각 값을 "그 행의 합"으로 나눠서, 합이 1이 되는 비율로 만듭니다.
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

In [ ]:
# 간단한 점수 배열로 softmax를 테스트해봅니다.
scores_1d = np.array([2.0, 1.0, 0.1])
weights_1d = softmax(scores_1d)

print("원래 점수:", scores_1d)
print("softmax 결과:", np.round(weights_1d, 3))
print("합계 (1이어야 함):", weights_1d.sum())
print()

# 2차원 배열(행이 여러 개)이라면 각 행마다 독립적으로 softmax가 적용됩니다.
scores_2d = np.array([
    [2.0, 1.0, 0.1],
    [1.0, 1.0, 1.0],   # 모든 점수가 같다면 → 가중치도 모두 동일(1/3)해야 합니다.
])
weights_2d = softmax(scores_2d)
print("2차원 입력에 대한 softmax 결과:\n", np.round(weights_2d, 3))
print("각 행의 합:", weights_2d.sum(axis=-1))

## 4. 아주 작은 예제로 직접 계산해보기

이제 3개의 "단어"를 4차원 벡터로 표현했다고 가정하고, Self-Attention 계산을 손으로 따라가듯 한 단계씩 실행해보겠습니다. 실제로는 이런 벡터(임베딩)를 학습을 통해 자동으로 얻지만, 여기서는 이해를 돕기 위해 **의미가 비슷한 단어일수록 벡터도 비슷하게** 직접 설계했습니다.

- 고양이(cat), 강아지(dog) → 둘 다 "동물"이라 벡터가 서로 비슷합니다.
- 자동차(car) → 동물과는 성격이 달라 벡터도 확연히 다릅니다.

또한 이번 절에서는 이해를 돕기 위해 Q, K, V를 만들 때 사용하는 가중치 행렬($W_Q, W_K, W_V$)을 잠시 생략하고, **Q = K = V = 입력 벡터 그대로** 사용합니다. (실제 모델에서는 이 벡터에 학습 가능한 가중치를 곱해서 Q/K/V를 "따로" 만드는데, 이 부분은 8절 Multi-Head Attention에서 다시 다룹니다.)

In [ ]:
#               [특징1, 특징2, 특징3, 특징4]
cat = np.array([1.0,  0.9,   0.1,  0.0])   # 고양이
dog = np.array([0.9,  1.0,   0.0,  0.1])   # 강아지 - 고양이와 벡터가 매우 비슷함
car = np.array([0.0,  0.1,   1.0,  0.9])   # 자동차 - 고양이/강아지와 벡터가 많이 다름

words = ["고양이(cat)", "강아지(dog)", "자동차(car)"]

# 세 벡터를 하나의 행렬로 쌓습니다. (단어 3개 x 4차원)
X_toy = np.stack([cat, dog, car])

print("X_toy shape:", X_toy.shape, " ← (단어 개수, 벡터 차원) = (3, 4)")
print(X_toy)

### Step 1. Query와 Key의 내적으로 "관련도 점수" 계산하기

이번 예제에서는 Q = K = X_toy 이므로, `X_toy @ X_toy.T`를 계산하면 됩니다.

- `X_toy`의 shape: (3, 4) → (단어 개수, 차원)
- `X_toy.T`의 shape: (4, 3)
- 곱한 결과의 shape: (3, 3) → **(단어 개수, 단어 개수)** 크기의 정사각행렬이 나옵니다.

즉 `scores[i, j]`는 "**i번째 단어**가 **j번째 단어**와 얼마나 관련 있는지"를 나타내는 점수입니다.

In [ ]:
d_k = X_toy.shape[-1]  # 벡터 하나의 차원 수 (여기서는 4)

raw_scores = X_toy @ X_toy.T
print("raw_scores shape:", raw_scores.shape, " ← (단어 개수, 단어 개수) = (3, 3)")
print(np.round(raw_scores, 3))

# 대각선(자기 자신과의 점수)이 가장 크고,
# 고양이-강아지 점수가 고양이-자동차 점수보다 훨씬 큰 것을 확인해보세요.

### Step 2. 왜 $\sqrt{d_k}$로 나눠줄까요? (스케일링)

벡터의 차원 수($d_k$)가 커지면, 내적 값도 함께 커지는 경향이 있습니다. 점수가 너무 커지면 softmax 결과가 한쪽으로 심하게 치우쳐서(예: [0.999, 0.0009, 0.0001]) 거의 "한 단어만 100% 선택"하는 극단적인 분포가 되어버립니다. 이렇게 되면 학습 시 기울기(gradient)가 거의 0에 가까워져서 모델이 잘 학습되지 않는 문제가 생깁니다.

그래서 점수를 $\sqrt{d_k}$로 나누어 값의 크기를 적당히 눌러주는데, 이를 **스케일링(scaling)** 이라고 부릅니다. (그래서 이 방식의 이름이 **Scaled** Dot-Product Attention입니다.)

In [ ]:
scaled_scores = raw_scores / np.sqrt(d_k)

print(f"d_k = {d_k}, sqrt(d_k) = {np.sqrt(d_k):.3f}")
print("scaled_scores:\n", np.round(scaled_scores, 3))

# raw_scores와 비교해보면 값의 "절대적인 크기"는 작아졌지만,
# 어떤 값이 더 큰지에 대한 순서(대소관계)는 그대로 유지된 것을 볼 수 있습니다.

### Step 3. softmax로 "가중치(비율)"로 바꾸기

이제 Step 2에서 구한 점수에, 앞서 만든 `softmax` 함수를 적용합니다. `axis=-1`(마지막 축, 즉 각 행)을 기준으로 적용하므로, **각 단어(행)마다 다른 단어들에 대한 가중치의 합이 1**이 됩니다.

In [ ]:
attention_weights = softmax(scaled_scores)

print("attention_weights:\n", np.round(attention_weights, 3))
print("\n각 행(단어)의 가중치 합:", attention_weights.sum(axis=-1), " ← 모두 1이어야 정상입니다.")

print("\n[해석] 고양이 행을 보면:")
for j, w in enumerate(words):
    print(f"  고양이 → {w} : {attention_weights[0, j]:.3f}")
print("→ 고양이는 자기 자신과 강아지에게는 비슷하게 큰 가중치를 주고,")
print("  자동차에게는 훨씬 작은 가중치를 주는 것을 확인할 수 있습니다.")

### Step 4. 가중치로 Value를 "가중합"해서 최종 출력 만들기

마지막으로, 위에서 구한 가중치(비율)를 Value(V)에 곱해서 더합니다. 즉 각 단어의 최종 출력은 "다른 모든 단어의 Value를, 자신이 계산한 가중치만큼 섞은 값"이 됩니다.

- `attention_weights` shape: (3, 3)
- `V`(=X_toy) shape: (3, 4)
- 곱한 결과 shape: (3, 4) → **입력과 동일한 shape**이 나옵니다. (단어 개수, 차원 수)

즉 출력에서 "고양이" 행은 원래의 고양이 벡터 그대로가 아니라, **고양이와 강아지의 정보가 많이 섞이고 자동차의 정보는 조금만 섞인** 새로운 벡터가 됩니다. 이것이 바로 "문맥을 반영한" 표현입니다.

In [ ]:
output_manual = attention_weights @ X_toy

print("output_manual shape:", output_manual.shape, " ← 입력 X_toy와 동일한 shape (3, 4)")
print(np.round(output_manual, 3))
print()
print("원래 고양이 벡터        :", np.round(cat, 3))
print("attention을 거친 고양이 :", np.round(output_manual[0], 3))
print("→ 강아지 쪽으로 값이 살짝 끌려온 반면, 자동차 방향으로는 거의 끌려오지 않았습니다.")

## 5. 지금까지의 4단계를 하나의 함수로 정리하기

Step 1~4에서 손으로 따라간 과정을 그대로 함수 하나로 합치면 다음과 같습니다. (원본 자료에는 이 함수에 들여쓰기 오류가 있어 실제로는 실행되지 않는 상태였습니다. 이번 버전에서는 오류를 수정하고, 각 줄이 위의 어느 단계와 대응되는지 주석을 달아 두었습니다.)

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention을 계산합니다.
    Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V

    Q: Query 행렬, shape = (단어 개수, d_k)
    K: Key 행렬,   shape = (단어 개수, d_k)
    V: Value 행렬, shape = (단어 개수, d_v)   (보통 d_v = d_k)
    mask: (단어 개수, 단어 개수) 크기의 0/1 행렬. 0인 위치는 "보면 안 되는" 위치입니다.
          None이면 마스킹을 적용하지 않습니다.

    반환값:
    output  : attention이 적용된 최종 결과, shape = (단어 개수, d_v)
    weights : 각 단어가 다른 단어에게 준 가중치(합이 1), shape = (단어 개수, 단어 개수)
    """
    d_k = Q.shape[-1]  # Step 2에서 나눠줄 스케일링 값을 위해 차원 수를 가져옵니다.

    # Step 1: Query와 Key의 내적으로 "관련도 점수" 계산
    #   Q shape: (n, d_k), K.T shape: (d_k, n) → scores shape: (n, n)
    # Step 2: sqrt(d_k)로 나눠서 값이 너무 커지지 않도록 스케일링
    scores = Q @ K.T / np.sqrt(d_k)

    # (선택) mask가 주어졌다면, 보면 안 되는 위치의 점수를 아주 작은 값(-1e9)으로 바꿉니다.
    # 아주 작은 값은 잠시 후 softmax를 거치면 사실상 0에 가까운 확률이 됩니다.
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)

    # Step 3: softmax로 점수를 "합이 1인 가중치"로 변환
    weights = softmax(scores)

    # Step 4: 가중치로 Value를 가중합 → 문맥이 반영된 최종 출력
    #   weights shape: (n, n), V shape: (n, d_v) → output shape: (n, d_v)
    output = weights @ V

    return output, weights


# 참고: 이 구현은 이해를 돕기 위해 "문장 하나(batch 없음)"만 처리하도록 단순화했습니다.
# 실제 PyTorch 등에서는 (batch, seq_len, d_k)처럼 배치 차원이 하나 더 있고,
# K.T 대신 K.transpose(-2, -1)처럼 "마지막 두 축만" 바꾸는 방식을 사용합니다.

In [ ]:
# 위에서 손으로 계산한 attention_weights, output_manual과 결과가 같은지 확인해봅니다.
output_fn, weights_fn = scaled_dot_product_attention(X_toy, X_toy, X_toy)

print("함수로 계산한 weights가 수동 계산 결과와 같은가?", np.allclose(weights_fn, attention_weights))
print("함수로 계산한 output이 수동 계산 결과와 같은가? ", np.allclose(output_fn, output_manual))

## 6. Masking: "미래를 보지 못하게" 막기

GPT와 같은 언어모델은 문장을 왼쪽에서 오른쪽으로 한 단어씩 예측합니다. 이때 다음 단어를 예측하면서 **아직 나오지 않은(미래) 단어를 미리 참고한다면 반칙**이겠죠? (마치 시험을 볼 때 답안지를 미리 보는 것과 같습니다.) 이를 막기 위해 사용하는 것이 **causal mask(순방향 마스크)** 입니다.

방법은 간단합니다.

1. "몇 번째 단어가 몇 번째 단어를 볼 수 있는지"를 나타내는 0/1 행렬(mask)을 만듭니다.
   - `mask[i, j] = 1` → i번째 단어가 j번째 단어를 봐도 된다
   - `mask[i, j] = 0` → i번째 단어가 j번째 단어를 보면 안 된다 (미래 토큰)
2. `scaled_dot_product_attention` 함수 안에서, mask가 0인 위치의 점수를 `-1e9`(거의 −무한대)로 바꿔줍니다.
3. softmax를 적용하면 $e^{-1e9}$는 사실상 0이 되므로, **그 위치의 가중치는 정확히 0에 가깝게** 됩니다.

아래에서 3개의 단어에 대해 "자기 자신과 그 이전 단어까지만 볼 수 있는" 마스크를 만들어 확인해보겠습니다.

In [ ]:
seq_len = X_toy.shape[0]  # 여기서는 3

# np.tril(하삼각행렬)을 쓰면 대각선을 포함한 아래쪽만 1, 위쪽은 0인 행렬을 쉽게 만들 수 있습니다.
causal_mask = np.tril(np.ones((seq_len, seq_len)))

print("causal_mask (1=볼 수 있음, 0=볼 수 없음):")
print(causal_mask)
print()
print("→ 0번째 단어(고양이)는 자기 자신만 볼 수 있고,")
print("  1번째 단어(강아지)는 0~1번째(고양이, 강아지)까지 볼 수 있고,")
print("  2번째 단어(자동차)는 전부(고양이, 강아지, 자동차) 볼 수 있습니다.")

In [ ]:
output_masked, weights_masked = scaled_dot_product_attention(X_toy, X_toy, X_toy, mask=causal_mask)

print("마스킹을 적용한 attention weights:")
print(np.round(weights_masked, 3))
print()
print("각 행의 합 (여전히 1이어야 합니다):", weights_masked.sum(axis=-1))
print()
print("[확인] 0번째 단어(고양이) 행에서 1,2번째(강아지, 자동차) 열의 값:",
      weights_masked[0, 1], weights_masked[0, 2])
print("→ 마스킹 전에는 0이 아니었던 값들이, 마스킹 후에는 정확히 0이 된 것을 확인할 수 있습니다.")

## 7. (보너스) 마스킹 전/후를 히트맵으로 비교하기

숫자만 보는 것보다 색으로 보면 "어떤 단어가 어떤 단어에 얼마나 집중하는지"를 더 직관적으로 파악할 수 있습니다. matplotlib으로 마스킹 전/후 attention weight를 나란히 그려보겠습니다. (matplotlib이 설치되어 있지 않다면 이 셀은 건너뛰어도 이후 실습에는 지장이 없습니다.)

> 참고: 한글 폰트가 설정되어 있지 않은 환경에서는 그래프 안의 한글이 네모(□)로 표시될 수 있어, 그래프의 축 라벨은 영어(cat/dog/car)로 표시했습니다.

In [ ]:
%matplotlib inline
# 위 magic command는 그래프가 노트북 안에 바로 표시되도록 설정합니다.
# (Jupyter/Colab에서는 보통 기본으로 켜져 있지만, 혹시 몰라 명시적으로 설정합니다.)

import matplotlib.pyplot as plt

words_en = ["cat", "dog", "car"]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

for ax, W, title in zip(
    axes,
    [attention_weights, weights_masked],
    ["Before masking (all visible)", "After causal masking"],
):
    im = ax.imshow(W, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(words_en)))
    ax.set_yticks(range(len(words_en)))
    ax.set_xticklabels(words_en)
    ax.set_yticklabels(words_en)
    ax.set_xlabel("Key")
    ax.set_ylabel("Query")
    ax.set_title(title)

    # 각 칸에 실제 숫자를 적어줍니다.
    for i in range(len(words_en)):
        for j in range(len(words_en)):
            ax.text(j, i, f"{W[i, j]:.2f}", ha="center", va="center",
                     color="white" if W[i, j] > 0.5 else "black")

fig.colorbar(im, ax=list(axes), fraction=0.046, pad=0.04, label="attention weight")
plt.show()

# 오른쪽 그래프(마스킹 후)의 오른쪽 위 삼각형 칸들이 전부 0.00(가장 옅은 색)인 것을
# 확인해보세요. → 이것이 "미래 단어를 보지 못하게 막았다"는 것이 시각적으로 나타난 모습입니다.

## 8. 왜 Head를 여러 개 사용할까요? (Multi-Head Attention)

지금까지 만든 것은 attention을 "한 번" 계산하는 방법입니다. 그런데 실제 Transformer는 이것을 **여러 개(예: 8개, 12개)를 동시에** 계산해서 합칩니다. 이를 **Multi-Head Attention**이라고 부릅니다.

왜 여러 번 계산할까요? 비유하자면, 한 문장을 여러 명의 서로 다른 전문가가 각자의 관점으로 동시에 읽는 것과 비슷합니다.

- 어떤 head는 "문법적으로 연결된 단어"에 집중할 수 있고,
- 다른 head는 "의미적으로 비슷한 단어"에 집중할 수 있고,
- 또 다른 head는 "가까운 위치의 단어"에 집중할 수도 있습니다.

각 head는 입력을 더 작은 차원($d_k = d_{model} / \text{head 개수}$)으로 나눠서 각자 독립적으로 attention을 계산한 뒤, 마지막에 모든 head의 결과를 이어붙이고(concatenate) 한 번 더 가중치 행렬($W_O$)을 곱해서 원래 차원($d_{model}$)으로 되돌립니다.

정리하면 구조는 다음과 같습니다.

```
입력 X (단어 개수, d_model)
   ├─ Head 0: X @ W_Q[0], X @ W_K[0], X @ W_V[0] → attention → (단어 개수, d_k)
   ├─ Head 1: X @ W_Q[1], X @ W_K[1], X @ W_V[1] → attention → (단어 개수, d_k)
   ├─ ...
   └─ Head (h-1): ...                                        → (단어 개수, d_k)
        │
        ▼ 전부 이어붙이기 (concatenate)
   (단어 개수, d_model)          ← head 개수 x d_k = d_model
        │
        ▼ W_O 를 곱해서 head들 사이 정보를 섞어줌
   출력 (단어 개수, d_model)      ← 입력과 동일한 shape!
```

In [ ]:
class MultiHeadAttention:
    """
    여러 개의 attention head를 병렬로 계산한 뒤 하나로 합치는
    Multi-Head Attention 구현입니다.

    d_model   : 입력/출력 벡터의 전체 차원 수 (예: 8)
    num_heads : head의 개수 (예: 2) — d_model은 반드시 num_heads로 나누어떨어져야 합니다.
    """

    def __init__(self, d_model, num_heads):
        # d_model이 num_heads로 딱 나누어떨어지지 않으면, 나중에 head들을
        # 다시 이어붙였을 때(concatenate) 크기가 d_model과 맞지 않게 되므로 미리 검사합니다.
        assert d_model % num_heads == 0, "d_model은 num_heads로 나누어떨어져야 합니다."

        self.d_k = d_model // num_heads   # head 하나가 담당하는 차원 수
        self.num_heads = num_heads

        # head마다 서로 다른(독립적인) Q/K/V 투영 행렬을 가집니다.
        # 각 행렬의 shape: (d_model, d_k)
        #   → X(단어 개수, d_model) @ W(d_model, d_k) = (단어 개수, d_k)
        # * 0.1을 곱하는 이유: 초기 가중치 값을 작게 만들어 학습 초반 값이 너무 크게
        #   튀지 않도록 하는 흔한 초기화 방식입니다. (실전에서는 Xavier/He 초기화 등을 사용합니다.)
        self.W_Q = [np.random.randn(d_model, self.d_k) * 0.1 for _ in range(num_heads)]
        self.W_K = [np.random.randn(d_model, self.d_k) * 0.1 for _ in range(num_heads)]
        self.W_V = [np.random.randn(d_model, self.d_k) * 0.1 for _ in range(num_heads)]

        # 모든 head의 결과를 이어붙인 뒤(shape: (단어 개수, d_model)) 다시 섞어주는 행렬입니다.
        self.W_O = np.random.randn(d_model, d_model) * 0.1

    def forward(self, X):
        """
        X: 입력 행렬, shape = (단어 개수, d_model)
        반환값: shape = (단어 개수, d_model)  ← 입력과 동일한 shape
                (그래야 여러 층을 계속 쌓아 올릴 수 있습니다)
        """
        heads = []  # 각 head의 출력을 담아둘 리스트

        for h in range(self.num_heads):
            # 1) 이 head 전용 Q, K, V를 만듭니다. 각각 shape: (단어 개수, d_k)
            Q = X @ self.W_Q[h]
            K = X @ self.W_K[h]
            V = X @ self.W_V[h]

            # 2) 이 head만의 attention을 계산합니다. head_out shape: (단어 개수, d_k)
            head_out, _ = scaled_dot_product_attention(Q, K, V)
            heads.append(head_out)

        # 3) 모든 head의 결과를 마지막 축(차원)을 기준으로 이어붙입니다.
        #    head마다 (단어 개수, d_k) → 전부 이어붙이면
        #    (단어 개수, num_heads * d_k) = (단어 개수, d_model)
        multi_head = np.concatenate(heads, axis=-1)

        # 4) W_O를 곱해 head들 사이의 정보를 한 번 더 섞어서 최종 출력을 만듭니다.
        return multi_head @ self.W_O

    def forward_with_shape_trace(self, X):
        """
        forward()와 완전히 동일하게 동작하지만, 각 단계의 shape을 출력해주는
        '학습/디버깅용' 버전입니다. 실전 코드에는 보통 넣지 않지만, 지금은
        데이터가 어떤 모양으로 흘러가는지 눈으로 직접 확인하기 위해 사용합니다.
        """
        print(f"[입력]    X shape            : {X.shape}   (단어 개수={X.shape[0]}, d_model={X.shape[1]})")

        heads = []
        for h in range(self.num_heads):
            Q = X @ self.W_Q[h]
            K = X @ self.W_K[h]
            V = X @ self.W_V[h]
            print(f"[Head {h}]  Q, K, V shape      : {Q.shape}   (단어 개수={Q.shape[0]}, d_k={Q.shape[1]})")

            head_out, weights = scaled_dot_product_attention(Q, K, V)
            print(f"[Head {h}]  attention weights  : {weights.shape}   (단어 개수 x 단어 개수)")
            print(f"[Head {h}]  head 출력 shape     : {head_out.shape}")

            heads.append(head_out)

        multi_head = np.concatenate(heads, axis=-1)
        print(f"[Concat]  전체 head를 이어붙인 shape : {multi_head.shape}   (단어 개수, num_heads*d_k = d_model)")

        output = multi_head @ self.W_O
        print(f"[출력]    W_O를 곱한 최종 shape      : {output.shape}")

        return output

### shape 변화를 눈으로 직접 추적해보기

`forward` 메서드는 결과만 돌려주기 때문에 중간 shape이 눈에 보이지 않습니다. 아래에서는 `forward_with_shape_trace` 메서드로, 입력 X가 각 단계를 지나며 shape이 어떻게 바뀌는지 직접 확인해보겠습니다.

In [ ]:
np.random.seed(42)  # 실행할 때마다 같은 랜덤 값이 나오도록 고정합니다 (재현성을 위한 좋은 습관입니다).

d_model = 8
num_heads = 2

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)

# 이번에는 5개의 단어로 이루어진, 8차원 임베딩을 가진 (가상의) 문장을 입력으로 사용합니다.
X = np.random.randn(5, d_model)

print("=== forward (일반 실행) ===")
output = mha.forward(X)
print(f"Multi-Head Attention: {X.shape} → {output.shape}")

print("\n=== forward_with_shape_trace (shape 추적) ===")
output_traced = mha.forward_with_shape_trace(X)

print("\n두 결과가 동일한가?", np.allclose(output, output_traced))

## 9. 정리 및 더 해볼 것들

이번 실습에서 다룬 내용을 정리하면 다음과 같습니다.

- **Self-Attention**은 문장 속 각 단어가 다른 단어를 "얼마나 참고할지" 가중치를 계산하고, 그 가중치만큼 정보를 섞어서 문맥이 반영된 새로운 표현을 만드는 방법입니다.
- 계산 과정은 `점수(QK^T) → 스케일링(÷√d_k) → softmax → 가중합(×V)` 네 단계로 이루어집니다.
- **Masking**은 특정 위치의 점수를 -1e9로 만들어, softmax 이후 해당 위치의 가중치가 사실상 0이 되도록 만드는 방법입니다. (예: 미래 토큰을 보지 못하게 막는 causal mask)
- **Multi-Head Attention**은 이 attention을 여러 head로 나누어 병렬로 계산한 뒤 합쳐서, 서로 다른 종류의 관계를 동시에 포착할 수 있게 해줍니다.

### 다음에 직접 바꿔서 실험해보세요

- `X_toy`의 자동차 벡터를 고양이/강아지와 더 비슷하게 바꾸면, attention weight가 어떻게 달라지나요?
- `d_model=8, num_heads=2`를 `num_heads=4`나 `num_heads=8`로 바꾸면 `d_k`는 어떻게 달라지나요? (`d_model % num_heads == 0` 조건을 깨는 값(예: num_heads=3)을 넣으면 어떤 에러가 나는지도 확인해보세요.)
- `causal_mask` 대신, 특정 단어 하나만 다른 모든 단어를 못 보게 막는 mask를 직접 만들어보세요.
- 문장의 단어 개수(`seq_len`)를 늘려서 attention weight 히트맵이 어떻게 바뀌는지 관찰해보세요.

### 참고: 실제 프레임워크와의 차이

이 노트북은 원리를 이해하기 위해 numpy로 "문장 하나"만 처리하도록 단순화했습니다. 실제 PyTorch 등에서는

- 여러 문장을 한꺼번에 처리하는 **배치(batch) 차원**이 추가되고,
- head별로 파이썬 `for`문을 도는 대신 하나의 큰 텐서 연산으로 한 번에 처리해서 훨씬 빠르며,
- 이번처럼 미리 정해둔 벡터가 아니라 실제 학습 데이터로부터 $W_Q, W_K, W_V, W_O$ 값을 **학습**하게 됩니다.

핵심 수식과 아이디어는 동일하니, 이 노트북에서 익힌 shape 감각을 그대로 가져가면 됩니다.